# LeetCode 2145 — Count the Hidden Sequences bằng Python

Link đề: https://leetcode.com/problems/count-the-hidden-sequences/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def numberOfArrays(self, differences: List[int], lower: int, upper: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra: biết **hiệu liên tiếp** thì cả dãy được xác định hoàn toàn bởi **phần tử đầu tiên** — `hidden[i] = hidden[0] + P[i]` với `P` là tổng tiền tố.
2. Biến bài đếm dãy thành **đếm số nguyên trong một đoạn**: giao của các ràng buộc `lower ≤ x + P[i] ≤ upper`.
3. Rút gọn còn **min và max của tiền tố** — một lượt, O(1) bộ nhớ.
4. Cẩn thận với **`max(…, 0)`** khi đoạn rỗng, và với tràn số ở ngôn ngữ khác (tiền tố tới 10¹⁰).
5. Kiểm chứng bằng vét cạn mọi giá trị đầu; mở rộng sang ràng buộc riêng cho từng vị trí.

## 2. Đề bài & ví dụ

Cho mảng `differences` độ dài `n` mô tả **hiệu** giữa các phần tử liên tiếp của một dãy ẩn `hidden` dài `n + 1`: `differences[i] = hidden[i+1] − hidden[i]`.

Cho thêm `lower`, `upper`: **mọi** phần tử của `hidden` phải nằm trong `[lower, upper]` (bao gồm hai đầu).

Trả về **số dãy ẩn** có thể có (0 nếu không có).

**Ví dụ 1:** `differences = [1,-3,4], lower = 1, upper = 6` → `2` (`[3,4,1,5]` và `[4,5,2,6]`).

**Ví dụ 2:** `differences = [3,-4,5,1,-2], lower = -4, upper = 5` → `4`.

**Ví dụ 3:** `differences = [4,-7,2], lower = 3, upper = 6` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `−10⁵ ≤ differences[i] ≤ 10⁵`, `−10⁵ ≤ lower ≤ upper ≤ 10⁵`.

In [1]:
from typing import List
from itertools import accumulate
import random, time

examples = [
    ([1, -3, 4], 1, 6, 2),
    ([3, -4, 5, 1, -2], -4, 5, 4),
    ([4, -7, 2], 3, 6, 0),
]
def rebuild(start, differences):
    seq = [start]
    for d in differences:
        seq.append(seq[-1] + d)
    return seq

print(rebuild(3, [1, -3, 4]), rebuild(4, [1, -3, 4]), rebuild(5, [1, -3, 4]))

[3, 4, 1, 5] [4, 5, 2, 6] [5, 6, 3, 7]


## 3. Kiến thức nền

### 3.1 Tổng tiền tố và "khôi phục dãy từ hiệu"

`P[0] = 0`, `P[i] = d[0] + … + d[i−1]`. Khi đó `hidden[i] = hidden[0] + P[i]`. `itertools.accumulate(d, initial=0)` cho đúng mảng `P`.

In [2]:
d = [1, -3, 4]
P = list(accumulate(d, initial=0))
print("P =", P, "-> hidden với hidden[0]=3:", [3 + p for p in P])

P = [0, 1, -2, 2] -> hidden với hidden[0]=3: [3, 4, 1, 5]


### 3.2 Đếm số nguyên trong đoạn `[a, b]`

Số nguyên `x` với `a ≤ x ≤ b` là `b − a + 1` nếu `a ≤ b`, ngược lại 0 ⇒ viết gọn `max(0, b − a + 1)`.

In [3]:
for a, b in [(3, 4), (5, 5), (6, 3)]:
    print(f"[{a}, {b}] có {max(0, b - a + 1)} số nguyên")

[3, 4] có 2 số nguyên
[5, 5] có 1 số nguyên
[6, 3] có 0 số nguyên


### 3.3 Giao nhiều đoạn

Giao của các đoạn `[a_i, b_i]` là `[max a_i, min b_i]` (rỗng nếu max > min).

In [4]:
segs = [(1, 9), (3, 12), (-2, 7)]
print("giao =", (max(a for a, _ in segs), min(b for _, b in segs)))

giao = (3, 7)


### 3.4 Giá trị lớn trong Python

`P[i]` có thể tới `10⁵ · 10⁵ = 10¹⁰` — vượt `int32` (C++/Java cần `long`). Python tự mở rộng số nguyên nên không cần lo, nhưng nên ý thức điều này khi dịch sang ngôn ngữ khác.

In [5]:
print(sum([10**5] * 10**5), "> 2^31 - 1 =", 2**31 - 1)

10000000000 > 2^31 - 1 = 2147483647


### 3.5 Mảng hiệu và tổng tiền tố là hai phép **ngược nhau**

Từ dãy `a` lấy hiệu liên tiếp được `d`; cộng dồn `d` (bắt đầu từ `a[0]`) lại được `a`. Mất thông tin duy nhất khi lấy hiệu là **giá trị đầu** `a[0]` — đó chính là "bậc tự do" mà bài này đếm.

In [6]:
a = [3, 4, 1, 5]
d = [a[i + 1] - a[i] for i in range(len(a) - 1)]
back = list(accumulate(d, initial=a[0]))
print("a =", a, "-> hiệu d =", d, "-> cộng dồn từ a[0]:", back, "| giống a:", back == a)
print("cộng dồn từ a[0] + 7:", list(accumulate(d, initial=a[0] + 7)), "(cùng hiệu, tịnh tiến 7)")

a = [3, 4, 1, 5] -> hiệu d = [1, -3, 4] -> cộng dồn từ a[0]: [3, 4, 1, 5] | giống a: True
cộng dồn từ a[0] + 7: [10, 11, 8, 12] (cùng hiệu, tịnh tiến 7)


## 4. Ý tưởng nền tảng / trực giác

Dãy hiệu cố định ⇒ "hình dáng" của dãy cố định; chỉ có thể **tịnh tiến** cả dãy lên/xuống bằng cách chọn `hidden[0] = x`.

Ví dụ 1: `P = [0, 1, −2, 2]`. Hình dáng (so với điểm đầu) thấp nhất là `−2`, cao nhất là `+2` ⇒ dãy "cao" `2 − (−2) = 4` đơn vị.

Khung cho phép `[1, 6]` cao `6 − 1 = 5` đơn vị. Dãy cao 4 đặt vào khung cao 5 ⇒ có `5 − 4 + 1 = 2` vị trí đặt (đáy dãy ở 1 hoặc 2).

Hình dung bằng bảng:

| x = hidden[0] | dãy | hợp lệ? |
|---|---|---|
| 2 | 2, 3, 0, 4 | không (0 < 1) |
| 3 | 3, 4, 1, 5 | ✔ |
| 4 | 4, 5, 2, 6 | ✔ |
| 5 | 5, 6, 3, 7 | không (7 > 6) |

⇒ Đáp án = `max(0, (upper − lower) − (maxP − minP) + 1)`.

**Làm tay thêm ví dụ 2** (`differences = [3,-4,5,1,-2]`, khung `[-4, 5]`):

| i | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| d[i−1] | — | 3 | −4 | 5 | 1 | −2 |
| P[i] | 0 | 3 | −1 | 4 | 5 | 3 |

`min P = −1`, `max P = 5` ⇒ dãy cao 6; khung cao `5 − (−4) = 9` ⇒ `9 − 6 + 1 = 4` dãy. ✔ (khớp 4 dãy liệt kê trong đề).

**Một cách nhìn khác:** "chiều cao" `max P − min P` chính là **hiệu lớn nhất giữa hai phần tử bất kỳ** của dãy ẩn — và nó không phụ thuộc `x`. Dãy chỉ "vừa" khung khi chiều cao ≤ độ rộng khung; phần dư `(upper − lower) − (cao)` là "độ rơ" cho phép tịnh tiến, cộng 1 vì tính cả vị trí sát đáy.

In [7]:
d, lo, hi = [1, -3, 4], 1, 6
P = list(accumulate(d, initial=0))
print("min P =", min(P), "max P =", max(P), "-> chiều cao dãy =", max(P) - min(P), "| chiều cao khung =", hi - lo)
for x in range(0, 7):
    seq = [x + p for p in P]
    print(x, seq, "OK" if all(lo <= v <= hi for v in seq) else "")

min P = -2 max P = 2 -> chiều cao dãy = 4 | chiều cao khung = 5
0 [0, 1, -2, 2] 
1 [1, 2, -1, 3] 
2 [2, 3, 0, 4] 
3 [3, 4, 1, 5] OK
4 [4, 5, 2, 6] OK
5 [5, 6, 3, 7] 
6 [6, 7, 4, 8] 


## 5. Cách 1 — Brute Force / baseline: thử mọi giá trị đầu

**Ý tưởng:** `hidden[0]` phải thuộc `[lower, upper]`. Thử từng giá trị, khôi phục dãy, kiểm tra mọi phần tử.

In [8]:
def brute(differences, lower, upper):
    cnt = 0
    for x in range(lower, upper + 1):
        v = x; ok = True
        for d in differences:
            v += d
            if v < lower or v > upper:
                ok = False; break
        cnt += ok
    return cnt

for d, lo, hi, exp in examples:
    got = brute(d, lo, hi)
    print(got, "OK" if got == exp else "SAI")

2 OK
4 OK
0 OK


**Độ phức tạp:** O((upper − lower + 1) · n) = tới `2·10⁵ · 10⁵ = 2·10¹⁰` ⇒ TLE.

**Nút thắt:** với mỗi `x` ta khôi phục lại cả dãy, trong khi các dãy chỉ khác nhau một hằng số tịnh tiến ⇒ điều kiện "mọi phần tử trong khung" chỉ phụ thuộc vào **phần tử nhỏ nhất và lớn nhất** của hình dáng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `hidden[i] = x + P[i]`. Điều kiện `lower ≤ x + P[i] ≤ upper` với mọi `i` ⇔

`lower − P[i] ≤ x ≤ upper − P[i]` với mọi `i` ⇔ `lower − min P ≤ x ≤ upper − max P`.

(Ràng buộc chặt nhất phía dưới đến từ `P` nhỏ nhất, phía trên từ `P` lớn nhất.) Số `x` nguyên:

`ans = max(0, (upper − max P) − (lower − min P) + 1) = max(0, (upper − lower) − (max P − min P) + 1)`.

**Bất biến khi quét:** sau khi xử lý `k` hiệu, `x_cur = P[k]`, `mi = min(P[0..k])`, `mx = max(P[0..k])`. Nhớ khởi tạo `mi = mx = 0` vì `P[0] = 0` (phần tử đầu cũng phải trong khung!).

**Dừng sớm:** nếu tại bất kỳ lúc nào `mx − mi > upper − lower` thì đáp án chắc chắn 0 (khoảng chỉ rộng thêm).

**Quy tắc "nếu… thì…" tóm tắt:**
- **Nếu** `x` (tiền tố hiện tại) nhỏ hơn `mi` **thì** đáy của hình dáng hạ xuống ⇒ cập nhật `mi`.
- **Nếu** `x` lớn hơn `mx` **thì** đỉnh nâng lên ⇒ cập nhật `mx`.
- **Nếu** chiều cao `mx − mi` vượt độ rộng khung **thì** không dãy nào vừa ⇒ 0.
- **Ngược lại** đáp án là độ rơ + 1.

Lưu ý `differences` có thể toàn số dương (dãy tăng dần) hoặc toàn âm (giảm dần) — khi đó `mi = 0` hoặc `mx = 0` suốt quá trình, công thức vẫn đúng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng tiền tố đầy đủ + giao các đoạn

In [9]:
def prefix_intersection(differences, lower, upper):
    P = list(accumulate(differences, initial=0))
    lo_x = max(lower - p for p in P)          # = lower - min(P)
    hi_x = min(upper - p for p in P)          # = upper - max(P)
    return max(0, hi_x - lo_x + 1)

for d, lo, hi, exp in examples:
    got = prefix_intersection(d, lo, hi)
    print(got, "OK" if got == exp else "SAI")

2 OK
4 OK
0 OK


**Độ phức tạp cách 2:** O(n) thời gian, O(n) bộ nhớ (mảng `P`). Viết đúng như định nghĩa "giao các đoạn" nên rất dễ tin.

### 7.2 Cách 3 — Một lượt, chỉ giữ min/max (cách doocs)

In [10]:
def one_pass(differences, lower, upper):
    x = mi = mx = 0
    for d in differences:
        x += d
        if x < mi: mi = x
        if x > mx: mx = x
    return max(0, (upper - lower) - (mx - mi) + 1)

for d, lo, hi, exp in examples:
    got = one_pass(d, lo, hi)
    print(got, "OK" if got == exp else "SAI")

2 OK
4 OK
0 OK


**Độ phức tạp cách 3:** O(n) thời gian, O(1) bộ nhớ.

### 7.3 Cách 4 — Một lượt + dừng sớm

Chiều cao `mx − mi` chỉ tăng; khi vượt `upper − lower` thì trả 0 ngay. Trong C++/Java điều này còn giúp **tránh tràn số** (không cần cộng dồn tiếp tới 10¹⁰) nếu dùng `int`.

In [11]:
def one_pass_early(differences, lower, upper):
    width = upper - lower
    x = mi = mx = 0
    for d in differences:
        x += d
        mi = min(mi, x); mx = max(mx, x)
        if mx - mi > width:
            return 0
    return width - (mx - mi) + 1

for d, lo, hi, exp in examples:
    got = one_pass_early(d, lo, hi)
    print(got, "OK" if got == exp else "SAI")

2 OK
4 OK
0 OK


Ở nhánh cuối không cần `max(0, …)` vì nếu đến được đó thì `mx − mi ≤ width`.

### 7.4 Cách 5 — Dùng `accumulate` + `min`/`max` dựng sẵn (ngắn nhất)

In [12]:
def builtin(differences, lower, upper):
    P = list(accumulate(differences, initial=0))
    return max(0, upper - lower - (max(P) - min(P)) + 1)

for d, lo, hi, exp in examples:
    print(builtin(d, lo, hi), "OK" if builtin(d, lo, hi) == exp else "SAI")

2 OK
4 OK
0 OK


`accumulate`, `min`, `max` chạy trong C ⇒ thường nhanh nhất trong Python dù tốn O(n) bộ nhớ.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 2: `differences = [3,-4,5,1,-2]`, `lower = -4`, `upper = 5` (`width = 9`).

| bước | d | x = P | mi | mx | mx − mi |
|---|---|---|---|---|---|
| 0 | — | 0 | 0 | 0 | 0 |
| 1 | 3 | 3 | 0 | 3 | 3 |
| 2 | −4 | −1 | −1 | 3 | 4 |
| 3 | 5 | 4 | −1 | 4 | 5 |
| 4 | 1 | 5 | −1 | 5 | 6 |
| 5 | −2 | 3 | −1 | 5 | 6 |

Đáp án `9 − 6 + 1 = 4`. Các giá trị đầu hợp lệ: `x ∈ [lower − mi, upper − mx] = [−3, 0]`.

In [13]:
def trace(differences, lower, upper):
    width = upper - lower; x = mi = mx = 0
    print(f"width = {width}")
    print(f"bước 0: x=0, mi=0, mx=0")
    for k, d in enumerate(differences, 1):
        x += d; mi = min(mi, x); mx = max(mx, x)
        print(f"bước {k}: d={d:>3}, x={x:>3}, mi={mi:>3}, mx={mx:>3}, chiều cao={mx - mi}")
        if mx - mi > width:
            print("chiều cao > width -> 0"); return 0
    ans = width - (mx - mi) + 1
    print(f"đáp án = {width} - {mx - mi} + 1 = {ans}; hidden[0] ∈ [{lower - mi}, {upper - mx}]")
    for s in range(lower - mi, upper - mx + 1):
        print("   ", rebuild(s, differences))
    return ans

trace([3, -4, 5, 1, -2], -4, 5)

width = 9
bước 0: x=0, mi=0, mx=0
bước 1: d=  3, x=  3, mi=  0, mx=  3, chiều cao=3
bước 2: d= -4, x= -1, mi= -1, mx=  3, chiều cao=4
bước 3: d=  5, x=  4, mi= -1, mx=  4, chiều cao=5
bước 4: d=  1, x=  5, mi= -1, mx=  5, chiều cao=6
bước 5: d= -2, x=  3, mi= -1, mx=  5, chiều cao=6
đáp án = 9 - 6 + 1 = 4; hidden[0] ∈ [-3, 0]
    [-3, 0, -4, 1, 2, 0]
    [-2, 1, -3, 2, 3, 1]
    [-1, 2, -2, 3, 4, 2]
    [0, 3, -1, 4, 5, 3]


4

Trace ví dụ 3 (dừng sớm):

In [14]:
trace([4, -7, 2], 3, 6)

width = 3
bước 0: x=0, mi=0, mx=0
bước 1: d=  4, x=  4, mi=  0, mx=  4, chiều cao=4
chiều cao > width -> 0


0

## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def numberOfArrays(self, differences: List[int], lower: int, upper: int) -> int:
        x = 0                 # x = P[i] = hidden[i] - hidden[0]
        mi = mx = 0           # P[0] = 0 cũng phải tính!
        for d in differences:
            x += d
            if x < mi:
                mi = x
            elif x > mx:
                mx = x
        return max(0, (upper - lower) - (mx - mi) + 1)

In [16]:
sol = Solution()
for d, lo, hi, exp in examples:
    got = sol.numberOfArrays(d, lo, hi)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

2 | kỳ vọng 2 | OK
4 | kỳ vọng 4 | OK
0 | kỳ vọng 0 | OK


Dùng `elif` được vì một giá trị `x` mới không thể vừa nhỏ hơn `mi` vừa lớn hơn `mx` (khi `mi ≤ 0 ≤ mx` luôn đúng).

## 10. Vì sao thuật toán đúng

**Song ánh:** mỗi dãy ẩn hợp lệ ↔ một giá trị đầu `x` hợp lệ. Thật vậy, dãy được xác định duy nhất bởi `x` (`hidden[i] = x + P[i]`), và hai `x` khác nhau cho hai dãy khác nhau (khác ngay phần tử đầu).

**Điều kiện hợp lệ của `x`:** `∀i: lower ≤ x + P[i] ≤ upper` ⇔ `x ≥ lower − P[i]` ∀i và `x ≤ upper − P[i]` ∀i ⇔ `x ≥ lower − min P` và `x ≤ upper − max P`.

**Đếm:** số nguyên trong `[lower − min P, upper − max P]` = `max(0, upper − max P − lower + min P + 1)`. Thuật toán tính đúng `min P`, `max P` (bất biến ở mục 6) ⇒ đúng. ∎

**Nhận xét:** vì `min P ≤ 0 ≤ max P` (do `P[0] = 0`), đoạn hợp lệ của `x` luôn nằm **bên trong** `[lower, upper]` — nên điều kiện "`hidden[0]` trong khung" đã được bao hàm, không cần kiểm tra riêng. Đây chính là lý do khởi tạo `mi = mx = 0` là bắt buộc: nó mã hóa ràng buộc của phần tử đầu.

## 11. Phân tích Time / Space Complexity

`n = len(differences)`, `W = upper − lower ≤ 2·10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi giá trị đầu | O(W·n) | O(1) |
| 2. Tiền tố + giao đoạn | O(n) | O(n) |
| 3. Một lượt min/max | O(n) | O(1) |
| 4. + dừng sớm | O(n), thường ít hơn | O(1) |
| 5. accumulate + built-in | O(n) | O(n) |

In [17]:
rng = random.Random(2145)
big = [rng.randint(-3, 3) for _ in range(10**5)]
for name, f in [("prefix_intersection", prefix_intersection), ("one_pass", one_pass),
                ("one_pass_early", one_pass_early), ("builtin", builtin), ("Solution", Solution().numberOfArrays)]:
    t = time.perf_counter(); r = f(big, -10**5, 10**5); dt = time.perf_counter() - t
    print(f"{name:<20} -> {r}  {dt*1000:.1f} ms")

prefix_intersection  -> 199449  8.3 ms
one_pass             -> 199449  3.1 ms
one_pass_early       -> 199449  6.1 ms
builtin              -> 199449  3.4 ms
Solution             -> 199449  3.0 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {"brute": brute, "prefix_intersection": prefix_intersection, "one_pass": one_pass,
       "one_pass_early": one_pass_early, "builtin": builtin, "Solution": Solution().numberOfArrays}
edge = examples + [
    ([0], 5, 5, 1),                  # dãy hằng, khung một điểm
    ([1], 5, 5, 0),                  # cần chiều cao 1, khung cao 0
    ([0, 0, 0], -3, 3, 7),
    ([10**5], -10**5, 10**5, 10**5 + 1),
    ([-10**5, 10**5], 0, 10**5, 1),
    ([1, 1, 1], 0, 2, 0),            # cao 3 > 2
    ([1, -1, 1, -1], 0, 1, 1),
    ([2, -1], 0, 1, 0),
]
for d, lo, hi, exp in edge:
    for name, f in ALL.items():
        if name == "brute" and hi - lo > 10**4: continue
        assert f(list(d), lo, hi) == exp, (name, d, lo, hi, f(d, lo, hi), exp)
print("Edge case OK:", len(edge))

Edge case OK: 11


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [19]:
rng = random.Random(42)
for _ in range(3000):
    n = rng.randint(1, 12)
    span = rng.choice([1, 3, 10, 50])
    d = [rng.randint(-span, span) for _ in range(n)]
    lo = rng.randint(-40, 40); hi = lo + rng.randint(0, 60)
    ref = brute(d, lo, hi)
    for name, f in ALL.items():
        assert f(d, lo, hi) == ref, (name, d, lo, hi)
print("3000 case ngẫu nhiên OK")

3000 case ngẫu nhiên OK


### 12.3 Kiểm tra bằng cách liệt kê các dãy thật (input vừa)

In [20]:
def all_hidden(differences, lower, upper):
    P = list(accumulate(differences, initial=0))
    return [[x + p for p in P] for x in range(lower - min(P), upper - max(P) + 1)]

rng = random.Random(7)
for _ in range(300):
    d = [rng.randint(-5, 5) for _ in range(rng.randint(1, 20))]; lo = rng.randint(-20, 0); hi = lo + rng.randint(0, 40)
    seqs = all_hidden(d, lo, hi)
    assert len(seqs) == Solution().numberOfArrays(d, lo, hi)
    for s in seqs:
        assert all(lo <= v <= hi for v in s) and all(s[i + 1] - s[i] == d[i] for i in range(len(d)))
print("liệt kê dãy thật: OK")

liệt kê dãy thật: OK


### 12.4 Trường hợp cực trị theo ràng buộc

- `n = 10⁵` hiệu đều bằng `10⁵` ⇒ chiều cao `10¹⁰` ≫ khung ⇒ 0 (và cách dừng sớm thoát ngay ở bước 2).
- Hiệu đan dấu `+10⁵, −10⁵, …` với khung `[−10⁵, 10⁵]` ⇒ chiều cao `10⁵` ⇒ `2·10⁵ − 10⁵ + 1 = 100001` dãy.

In [21]:
inc = [10**5] * 10**5
alt = [10**5 if i % 2 == 0 else -10**5 for i in range(10**5)]
for name, f in ALL.items():
    if name == "brute": continue
    assert f(inc, -10**5, 10**5) == 0
    assert f(alt, -10**5, 10**5) == 100001
print("cực trị OK")

cực trị OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên `P[0] = 0` (khởi tạo `mi`, `mx` bằng phần tử tiền tố đầu tiên)

In [22]:
def wrong_init(differences, lower, upper):
    x = differences[0]; mi = mx = x                  # SAI: bỏ qua hidden[0] (P[0] = 0)
    for d in differences[1:]:
        x += d; mi = min(mi, x); mx = max(mx, x)
    return max(0, (upper - lower) - (mx - mi) + 1)

print("bản sai :", wrong_init([1, -3, 4], 1, 6))
print("bản đúng:", Solution().numberOfArrays([1, -3, 4], 1, 6))
print("bản sai :", wrong_init([5], 0, 3), "| bản đúng:", Solution().numberOfArrays([5], 0, 3))

bản sai : 2
bản đúng: 2
bản sai : 4 | bản đúng: 0


Ví dụ 1 tình cờ vẫn đúng (vì `P[0] = 0` nằm giữa min và max của các tiền tố còn lại), nhưng `[5]` với khung `[0, 3]` lộ lỗi: dãy `x, x + 5` cần khung cao ít nhất 5, bản sai lại nghĩ chiều cao là 0.

**Vì sao sai:** phần tử đầu `hidden[0]` (ứng với `P[0] = 0`) cũng phải nằm trong khung. Bỏ nó làm "chiều cao" dãy bị đánh giá thấp. **Sửa:** `x = mi = mx = 0` rồi mới duyệt.

### Lỗi 2 — Quên `max(0, …)` (trả số âm)

In [23]:
def wrong_no_clamp(differences, lower, upper):
    x = mi = mx = 0
    for d in differences:
        x += d; mi = min(mi, x); mx = max(mx, x)
    return (upper - lower) - (mx - mi) + 1                # SAI

print("bản sai :", wrong_no_clamp([4, -7, 2], 3, 6))
print("bản đúng:", Solution().numberOfArrays([4, -7, 2], 3, 6))

bản sai : -3
bản đúng: 0


**Sửa:** số lượng không thể âm ⇒ `max(0, …)`.

### Lỗi 3 — Quên `+ 1` (đếm độ dài đoạn thay vì số điểm nguyên)

In [24]:
def wrong_off_by_one(differences, lower, upper):
    x = mi = mx = 0
    for d in differences:
        x += d; mi = min(mi, x); mx = max(mx, x)
    return max(0, (upper - lower) - (mx - mi))            # SAI: thiếu +1

print("bản sai :", wrong_off_by_one([1, -3, 4], 1, 6))
print("bản đúng:", Solution().numberOfArrays([1, -3, 4], 1, 6))

bản sai : 1
bản đúng: 2


**Sửa:** đoạn `[a, b]` có `b − a + 1` số nguyên (bao gồm cả hai đầu).

### Lỗi 4 — Chỉ dùng tổng cuối cùng (bỏ qua các điểm giữa)

In [25]:
def wrong_last_only(differences, lower, upper):
    total = sum(differences)                              # SAI: chỉ xét hidden[0] và hidden[n]
    lo_x = max(lower, lower - total); hi_x = min(upper, upper - total)
    return max(0, hi_x - lo_x + 1)

print("bản sai :", wrong_last_only([1, -3, 4], 1, 6))
print("bản đúng:", Solution().numberOfArrays([1, -3, 4], 1, 6))

bản sai : 4
bản đúng: 2


**Vì sao sai:** dãy có thể "lặn" xuống thấp hoặc "vọt" lên cao ở giữa (ví dụ 1 xuống −2 rồi lên +2). Phải xét **mọi** tiền tố. **Sửa:** theo dõi min/max qua toàn bộ quá trình.

### Lỗi 5 — Brute force với khoảng lớn (TLE)

In [26]:
d = [rng.randint(-1, 1) for _ in range(1000)]
t = time.perf_counter(); r1 = brute(d, -5000, 5000); t1 = time.perf_counter() - t
t = time.perf_counter(); r2 = Solution().numberOfArrays(d, -5000, 5000); t2 = time.perf_counter() - t
print(f"cùng kết quả {r1 == r2}: brute {t1*1000:.0f} ms vs O(n) {t2*1000:.2f} ms  (đề thật: khoảng 2e5, n 1e5 -> brute ~1e10 phép)")

cùng kết quả True: brute 288 ms vs O(n) 0.07 ms  (đề thật: khoảng 2e5, n 1e5 -> brute ~1e10 phép)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cho **hiệu** liên tiếp" ⇒ dãy = **hằng số + tổng tiền tố**; mọi lời giải chỉ khác nhau bởi một phép **tịnh tiến**.
- Ràng buộc "mọi phần tử trong `[L, U]`" với dãy tịnh tiến được ⇒ chỉ cần **min và max** của tiền tố.
- Đếm số nguyên trong đoạn: `max(0, b − a + 1)`.
- Bài cùng họ: 1413 Minimum Value to Get Positive Step by Step Sum (tìm giá trị đầu nhỏ nhất), 1094 Car Pooling, 370 Range Addition (mảng hiệu), 2381.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về dãy ẩn **nhỏ nhất** (theo thứ tự từ điển) hoặc `[]`

Dãy nhỏ nhất ⇔ `hidden[0]` nhỏ nhất ⇔ `x = lower − min P`.

In [27]:
def smallest_hidden(differences, lower, upper):
    P = list(accumulate(differences, initial=0))
    x = lower - min(P)
    if x > upper - max(P): return []
    return [x + p for p in P]

for d, lo, hi, _ in examples:
    print(d, lo, hi, "->", smallest_hidden(d, lo, hi))

[1, -3, 4] 1 6 -> [3, 4, 1, 5]
[3, -4, 5, 1, -2] -4 5 -> [-3, 0, -4, 1, 2, 0]
[4, -7, 2] 3 6 -> []


### 15.2 Biến thể B — Mỗi vị trí có khung riêng `[lo_i, hi_i]`

Điều kiện `lo_i ≤ x + P[i] ≤ hi_i` ⇒ `x ∈ [max(lo_i − P[i]), min(hi_i − P[i])]`. Vẫn O(n).

In [28]:
def count_with_bounds(differences, lows, highs):
    P = list(accumulate(differences, initial=0))
    a = max(l - p for l, p in zip(lows, P)); b = min(h - p for h, p in zip(highs, P))
    return max(0, b - a + 1)

def count_with_bounds_brute(differences, lows, highs):
    cnt = 0
    for x in range(min(lows) - 200, max(highs) + 201):
        s = rebuild(x, differences)
        cnt += all(l <= v <= h for v, l, h in zip(s, lows, highs))
    return cnt

print(count_with_bounds([1, -3, 4], [1, 1, 1, 1], [6, 6, 6, 6]), count_with_bounds([1, -3, 4], [3, 1, 1, 1], [6, 6, 6, 5]))
rng = random.Random(152)
for _ in range(400):
    n = rng.randint(1, 6); d = [rng.randint(-4, 4) for _ in range(n)]
    lows = [rng.randint(-10, 5) for _ in range(n + 1)]; highs = [l + rng.randint(0, 12) for l in lows]
    assert count_with_bounds(d, lows, highs) == count_with_bounds_brute(d, lows, highs)
print("khớp brute force")

2 1
khớp brute force


### 15.3 Biến thể C — Một số hiệu bị **mất** (`None`), hiệu mất có thể là bất kỳ số nguyên trong `[-k, k]`

Đếm số dãy hợp lệ bây giờ phụ thuộc vào cả các hiệu mất ⇒ DP trên giá trị hiện tại: `ways[v]` = số cách để phần tử hiện tại bằng `v`. O(n · W · k) — chỉ cho W nhỏ. Minh họa sự khác biệt về độ khó khi bỏ đi tính "tịnh tiến".

In [29]:
def count_with_missing(differences, lower, upper, k):
    W = upper - lower + 1
    ways = [1] * W                                     # hidden[0] bất kỳ trong khung
    for d in differences:
        nw = [0] * W
        for v in range(W):
            if not ways[v]: continue
            steps = [d] if d is not None else range(-k, k + 1)
            for s in steps:
                u = v + s
                if 0 <= u < W: nw[u] += ways[v]
        ways = nw
    return sum(ways)

print(count_with_missing([1, -3, 4], 1, 6, 0))          # không có hiệu mất: = bài gốc (2)
print(count_with_missing([1, None, 4], 1, 6, 3))

2
7


### 15.4 Biến thể D — LC 1413: giá trị đầu **nhỏ nhất** để mọi tổng tiền tố (tính cả giá trị đầu) ≥ 1

Cùng ý tưởng "tịnh tiến": cần `x + P[i] ≥ 1` với mọi `i ≥ 1` ⇒ `x ≥ 1 − min_{i≥1} P[i]`, và `x ≥ 1`.

In [30]:
def min_start_value(nums):
    run = 0; low = 0
    for v in nums:
        run += v; low = min(low, run)
    return max(1, 1 - low)

def min_start_value_brute(nums):
    x = 1
    while True:
        run = x
        if all((run := run + v) >= 1 for v in nums): return x
        x += 1

print(min_start_value([-3, 2, -3, 4, 2]), min_start_value([1, 2]), min_start_value([1, -2, -3]))
rng = random.Random(154)
for _ in range(500):
    nums = [rng.randint(-10, 10) for _ in range(rng.randint(1, 10))]
    assert min_start_value(nums) == min_start_value_brute(nums)
print("khớp brute force")

5 1 5
khớp brute force


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi giá trị đầu | khôi phục và kiểm tra | O(W·n) | O(1) | kiểm chứng |
| 2. Tiền tố + giao đoạn | `x ∈ ∩ [lower−P_i, upper−P_i]` | O(n) | O(n) | dễ mở rộng (biến thể B) |
| 3. Một lượt min/max | công thức chiều cao | O(n) | O(1) | **mặc định** |
| 4. + dừng sớm | dừng khi cao > W | O(n) | O(1) | tránh tràn số ở C++/Java |
| 5. Built-in | `accumulate`, `min`, `max` | O(n) | O(n) | code ngắn nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `numberOfArrays([-40], -46, 53)` và `numberOfArrays([2, 2, -4], 0, 3)`?

**Bài 2 (tính tay):** Với `differences = [-2, 5, -1, -3]`, tính `P`, `min P`, `max P`. Nếu `lower = 0, upper = 10`, các giá trị `hidden[0]` hợp lệ là gì?

**Bài 3 (điền TODO):** Viết `bt3_todo(differences, lower, upper)` trả về **đoạn** `(a, b)` các giá trị `hidden[0]` hợp lệ, hoặc `None`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(differences, lower, upper)` cài cách 4 (một lượt + dừng sớm).

**Bài 5 (mini challenge):** Viết `bt5_todo(differences, lower)` trả về giá trị `upper` **nhỏ nhất** sao cho có ít nhất một dãy ẩn (giữ nguyên `lower`).

In [31]:
def bt3_todo(differences, lower, upper):
    # TODO: đoạn (a, b) của hidden[0] hợp lệ hoặc None
    pass

def bt4_todo(differences, lower, upper):
    # TODO: một lượt + dừng sớm
    pass

def bt5_todo(differences, lower):
    # TODO: upper nhỏ nhất để có ít nhất một dãy
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính chiều cao dãy. Mức 2: `[-40]`: P = [0, −40], cao 40, khung cao 99. Mức 3: `99 − 40 + 1 = 60`; `[2,2,-4]`: P = [0,2,4,0], cao 4 > 3 ⇒ `0`.

**Bài 2.** Mức 1: P = [0, −2, 3, 2, −1]. Mức 2: min = −2, max = 3. Mức 3: `x ∈ [0 − (−2), 10 − 3] = [2, 7]` ⇒ 6 giá trị.

**Bài 3.** Mức 1: `a = lower − min P`, `b = upper − max P`. Mức 2: rỗng nếu `a > b`. Mức 3: trả `(a, b)` hoặc `None`.

**Bài 4.** Mức 1: `width = upper − lower`. Mức 2: cập nhật `mi`, `mx` mỗi bước. Mức 3: `if mx − mi > width: return 0`.

**Bài 5.** Mức 1: cần khung cao ít nhất bằng chiều cao dãy. Mức 2: `upper − lower ≥ max P − min P`. Mức 3: `upper = lower + (max P − min P)`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `60`, `0`. **Bài 2:** `P = [0, −2, 3, 2, −1]`, `min = −2`, `max = 3`; `hidden[0] ∈ [2, 7]`.

In [32]:
assert Solution().numberOfArrays([-40], -46, 53) == 60
assert Solution().numberOfArrays([2, 2, -4], 0, 3) == 0
P = list(accumulate([-2, 5, -1, -3], initial=0))
assert P == [0, -2, 3, 2, -1] and (0 - min(P), 10 - max(P)) == (2, 7)

def bt3(differences, lower, upper):
    P = list(accumulate(differences, initial=0))
    a, b = lower - min(P), upper - max(P)
    return (a, b) if a <= b else None

def bt4(differences, lower, upper):
    width = upper - lower; x = mi = mx = 0
    for d in differences:
        x += d
        if x < mi: mi = x
        if x > mx: mx = x
        if mx - mi > width: return 0
    return width - (mx - mi) + 1

def bt5(differences, lower):
    P = list(accumulate(differences, initial=0))
    return lower + max(P) - min(P)

rng = random.Random(1945)
for _ in range(2000):
    d = [rng.randint(-6, 6) for _ in range(rng.randint(1, 10))]
    lo = rng.randint(-20, 20); hi = lo + rng.randint(0, 30)
    ref = brute(d, lo, hi)
    assert bt4(d, lo, hi) == ref
    seg = bt3(d, lo, hi)
    assert (0 if seg is None else seg[1] - seg[0] + 1) == ref
    u = bt5(d, lo)
    assert brute(d, lo, u) >= 1 and (u == lo or brute(d, lo, u - 1) == 0)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
x = mi = mx = 0              # P[0] = 0
for d in differences:
    x += d
    mi = min(mi, x)
    mx = max(mx, x)
return max(0, (upper - lower) - (mx - mi) + 1)
```

**Checklist:**
- [ ] Khởi tạo `mi = mx = 0` (phần tử đầu cũng phải trong khung).
- [ ] `+ 1` khi đếm số nguyên trong đoạn.
- [ ] `max(0, …)` khi không có dãy nào.
- [ ] Xét **mọi** tiền tố, không chỉ tổng cuối.
- [ ] (C++/Java) tiền tố tới 10¹⁰ ⇒ `long`, hoặc dừng sớm.